# MammosighTR — Çoklu Mimari Hibrit Topluluk Öğrenmesi (Ensemble) ve Grad-CAM Açıklanabilirlik Analizi

Bu çalışma, mamografi sınıflandırma görevinde en yüksek tanısal performansı elde etmek amacıyla ConvNeXt-Small, YOLO11m-cls ve EfficientNet-B0 mimarilerinin olasılık çıktılarının ağırlıklı topluluk (ensemble) füzyonunu ve Grad-CAM açıklanabilirlik analizini gerçekleştirmektedir.

### Metodoloji:
1. **Topluluk Modelleri:** ConvNeXt-Small (GeM + Kalite MLP), YOLO11m-cls ve EfficientNet-B0 çıkarımlarının optimal ağırlık optimizasyonu.
2. **Bağlamsal Füzyon:** Hasta bazlı 4-yön (RCC, LCC, RMLO, LMLO) bağlamsal modelleme ve klinik kalibrasyon.
3. **Açıklanabilir Yapay Zeka (XAI):** Sınıflandırma kararlarının Grad-CAM aktivasyon haritaları üzerinden anatomik doğrulaması.


In [ ]:
# ==============================================================================
# Bölüm 1: Ortam Konfigürasyonu, Bağımlılıklar ve Başlatma
# ==============================================================================
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'timm', 'albumentations', 'scikit-learn', 'scipy'], check=False)

import os, re, glob, time, json, random, shutil, zipfile
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.optimize import minimize

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import timm
from timm.utils import ModelEmaV2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from ultralytics import YOLO

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    roc_auc_score, f1_score, accuracy_score
)
from sklearn.preprocessing import label_binarize

SESSION_START_TIME = time.time()
MAX_SESSION_SECONDS = int(11.2 * 3600)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_GPUS = torch.cuda.device_count()
print(f'PyTorch: {torch.__version__} | Cihaz: {DEVICE} ({NUM_GPUS} GPU)')


In [ ]:
# ── 2. Veri Seti ve Önceki Oturum Ağırlıklarını / Önbelleklerini Kurtarma ──
DATASET_DIR     = '/kaggle/temp/yolo_dataset' if os.path.exists('/kaggle') else './yolo_dataset'
SAVE_DIR        = '/kaggle/working/runs/sota_ensemble' if os.path.exists('/kaggle') else './runs/sota_ensemble'
CONVNEXT_PT     = os.path.join(SAVE_DIR, 'best_convnext_small.pth')
LAST_CKPT       = os.path.join(SAVE_DIR, 'last_convnext_checkpoint.pth')
RESULTS_CSV     = os.path.join(SAVE_DIR, 'convnext_results.csv')
STATE_JSON      = os.path.join(SAVE_DIR, 'convnext_state.json')
Q_CACHE_NPZ     = os.path.join(SAVE_DIR, 'quality_features_cache.npz')
ENS_CACHE_NPZ   = os.path.join(SAVE_DIR, 'ensemble_tta_cache.npz')
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

if os.path.exists('/kaggle/input'):
    for fname, dst_path in [
        ('best_convnext_small.pth', CONVNEXT_PT),
        ('last_convnext_checkpoint.pth', LAST_CKPT),
        ('convnext_results.csv', RESULTS_CSV),
        ('convnext_state.json', STATE_JSON),
        ('quality_features_cache.npz', Q_CACHE_NPZ),
        ('ensemble_tta_cache.npz', ENS_CACHE_NPZ),
    ]:
        if not os.path.exists(dst_path):
            matches = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
            if matches:
                shutil.copy2(matches[0], dst_path)
                print(f'Önceki oturumdan geri yüklendi: {matches[0]} -> {dst_path}')

if not os.path.exists(os.path.join(DATASET_DIR, 'train')):
    zip_Search = (
        glob.glob('/kaggle/input/**/yolo_dataset.zip', recursive=True) +
        glob.glob('/kaggle/working/**/yolo_dataset.zip', recursive=True) +
        glob.glob('./**/yolo_dataset.zip', recursive=True)
    )
    if zip_Search:
        with zipfile.ZipFile(zip_Search[0], 'r') as zf:
            zf.extractall(DATASET_DIR)
        if not os.path.exists(os.path.join(DATASET_DIR, 'train')):
            sub_trains = glob.glob(os.path.join(DATASET_DIR, '*/train'))
            if sub_trains:
                DATASET_DIR = os.path.dirname(sub_trains[0])
        print(f'yolo_dataset.zip çıkartıldı. Aktif dizin: {DATASET_DIR}')
    else:
        dir_Search = [d for d in glob.glob('/kaggle/input/**/yolo_dataset', recursive=True) if os.path.isdir(d)]
        if dir_Search:
            DATASET_DIR = dir_Search[0]
        else:
            raise FileNotFoundError('yolo_dataset.zip bulunamadı!')

for split in ['train', 'val', 'test']:
    split_path = os.path.join(DATASET_DIR, split)
    for cls_Old, cls_New in [('BIRADS0', 'BIRADS_0'), ('BIRADS1_2', 'BIRADS_1_2'), ('BIRADS4_5', 'BIRADS_4_5')]:
        p_old = os.path.join(split_path, cls_Old)
        p_new = os.path.join(split_path, cls_New)
        if os.path.exists(p_old) and not os.path.exists(p_new):
            os.rename(p_old, p_new)

CLASS_NAMES   = ['BIRADS_0', 'BIRADS_1_2', 'BIRADS_4_5']
DISPLAY_NAMES = ['BI-RADS 0 (Ek Tetkik)', 'BI-RADS 1-2 (Normal/Benign)', 'BI-RADS 4-5 (Şüpheli)']
for split in ['train', 'val', 'test']:
    counts = {cls: len(glob.glob(f'{DATASET_DIR}/{split}/{cls}/*.jpg')) for cls in CLASS_NAMES}
    print(f'{split:5s}: {counts} (Toplam: {sum(counts.values())})')


In [ ]:
# ==============================================================================
# Bölüm 3: ConvNeXt-Small (GeM + Kalite MLP) Mimarisi ve Eğitimi
# ==============================================================================
IMG_SIZE   = 512
BATCH_SIZE = 24 * max(1, NUM_GPUS)

def extract_quality_positioning_features(img_gray):
    """12-boyutlu görüntü kalitesi ve anatomik pozisyonlama öznitelik vektörü çıkarımı.

    Öznitelikler: Yoğunluk istatistikleri (ortalama, varyans, entropi), yerel kontrast,
    Sobel kenar gradyanları, Laplacian keskinlik skoru, kütle merkezi ve doku doluluk oranı.
    """
    h, w = img_gray.shape[:2]
    fg = img_gray > 5
    fg_ratio = float(fg.mean())
    if fg_ratio < 0.01:
        return np.zeros(12, dtype=np.float32)
    fg_pixels = img_gray[fg].astype(np.float32) / 255.0
    mean_int = float(fg_pixels.mean()); std_int = float(fg_pixels.std())
    p90_int  = float(np.percentile(fg_pixels, 90))
    dense_ratio = float((fg_pixels > 0.65).mean())
    hist = cv2.calcHist([img_gray], [0], fg.astype(np.uint8), [32], [1, 256]).flatten()
    hist = hist / (hist.sum() + 1e-7)
    entropy = float(-np.sum(hist * np.log2(hist + 1e-7)) / 5.0)
    lap_var = float(np.log1p(cv2.Laplacian(img_gray, cv2.CV_32F)[fg].var()) / 10.0)
    gx = cv2.Sobel(img_gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(img_gray, cv2.CV_32F, 0, 1, ksize=3)
    edge_mean = float(np.log1p(np.hypot(gx, gy)[fg].mean()) / 6.0)
    chest_wall = max(float(fg[:, :max(1, int(w * 0.08))].mean()), float(fg[:, min(w - 1, int(w * 0.92)):].mean()))
    upper_q = float(fg[:int(h * 0.35), :].mean()); lower_q = float(fg[int(h * 0.65):, :].mean())
    ys, xs = np.where(fg)
    bbox_fill = float(fg_ratio / max(((ys.max() - ys.min() + 1) / float(h)) * ((xs.max() - xs.min() + 1) / float(w)), 1e-4))
    return np.array([fg_ratio, mean_int, std_int, p90_int, dense_ratio, entropy,
                     lap_var, edge_mean, chest_wall, upper_q, lower_q, bbox_fill], dtype=np.float32)

def scan_split_files(root_dir):
    samples, labels, cases, views = [], [], [], []
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        cls_dir = os.path.join(root_dir, cls_name)
        if not os.path.exists(cls_dir): continue
        for fpath in sorted(glob.glob(os.path.join(cls_dir, '*.jpg'))):
            bname = os.path.basename(fpath)
            m_case = re.search(r'case(\d+)', bname)
            m_view = re.search(r'_(RCC|LCC|RMLO|LMLO)_', bname)
            samples.append(fpath)
            labels.append(cls_idx)
            cases.append(int(m_case.group(1)) if m_case else -1)
            views.append(m_view.group(1) if m_view else 'UNK')
    return samples, np.array(labels, dtype=np.int64), np.array(cases, dtype=np.int64), np.array(views)

train_samples, train_labels, train_cases, train_views = scan_split_files(f'{DATASET_DIR}/train')
val_samples, val_labels, val_cases, val_views         = scan_split_files(f'{DATASET_DIR}/val')
test_samples, test_labels, test_cases, test_views     = scan_split_files(f'{DATASET_DIR}/test')

if os.path.exists(Q_CACHE_NPZ):
    q_data = np.load(Q_CACHE_NPZ)
    if len(train_samples) > 0 and len(q_data['train_q']) == len(train_samples):
        train_q, val_q, test_q = q_data['train_q'], q_data['val_q'], q_data['test_q']
        print(f'12-B Kalite öznitelikleri önbellekten yüklendi: {Q_CACHE_NPZ}')
    else:
        os.remove(Q_CACHE_NPZ)

if not os.path.exists(Q_CACHE_NPZ):
    print('12-B Kalite öznitelikleri hesaplanıyor ve önbelleğe kaydediliyor...')
    def _compute_q_list(fpaths, desc):
        arr = np.zeros((len(fpaths), 12), dtype=np.float32)
        for i, fp in enumerate(tqdm(fpaths, desc=desc, leave=False)):
            im = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)
            if im is not None: arr[i] = extract_quality_positioning_features(im)
        return arr
    train_q = _compute_q_list(train_samples, 'Train Q-Features')
    val_q   = _compute_q_list(val_samples,   'Val Q-Features')
    test_q  = _compute_q_list(test_samples,  'Test Q-Features')
    np.savez_compressed(Q_CACHE_NPZ, train_q=train_q, val_q=val_q, test_q=test_q)

class MammosightCachedDataset(Dataset):
    def __init__(self, samples, labels, q_feats, transform=None):
        self.samples, self.labels, self.q_feats, self.transform = samples, labels, q_feats, transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        img = cv2.imread(self.samples[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None else np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        if self.transform: img = self.transform(image=img)['image']
        return img, torch.tensor(self.q_feats[idx], dtype=torch.float32), self.labels[idx]

train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE), A.HorizontalFlip(p=0.5),
    A.Affine(scale=(0.92, 1.08), translate_percent=(-0.05, 0.05), rotate=(-12, 12), border_mode=cv2.BORDER_CONSTANT, p=0.6),
    A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2()
])
val_tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2()])

train_ds = MammosightCachedDataset(train_samples, train_labels, train_q, train_tf)
val_ds   = MammosightCachedDataset(val_samples,   val_labels,   val_q,   val_tf)
test_ds  = MammosightCachedDataset(test_samples,  test_labels,  test_q,  val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p); self.eps = eps
    def forward(self, x):
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p), (x.size(-2), x.size(-1))).pow(1.0 / self.p)

class MammosightConvNeXt(nn.Module):
    def __init__(self, model_name='convnext_small.fb_in22k_ft_in1k_384', num_classes=3, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool='', drop_path_rate=0.25)
        in_features = self.backbone.num_features
        self.gem = GeM(p=3.0)
        self.maxpool = nn.AdaptiveMaxPool2d(1)
        self.quality_mlp = nn.Sequential(
            nn.Linear(12, 32), nn.BatchNorm1d(32), nn.SiLU(inplace=True),
            nn.Linear(32, 32), nn.BatchNorm1d(32), nn.SiLU(inplace=True),
        )
        self.neck = nn.Sequential(
            nn.Linear(in_features * 2 + 32, 512), nn.BatchNorm1d(512), nn.SiLU(inplace=True),
        )
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in [0.2, 0.3, 0.4, 0.5]])
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x, q_feat):
        feat_map = self.backbone(x)
        f_gem = self.gem(feat_map).flatten(1)
        f_max = self.maxpool(feat_map).flatten(1)
        f_q   = self.quality_mlp(q_feat)
        emb   = self.neck(torch.cat([f_gem, f_max, f_q], dim=1))
        if self.training:
            return torch.stack([self.fc(d(emb)) for d in self.dropouts], dim=0).mean(dim=0)
        return self.fc(emb)

convnext_model = MammosightConvNeXt(pretrained=not os.path.exists(CONVNEXT_PT)).to(DEVICE)

is_convnext_done = False
if os.path.exists(STATE_JSON) and os.path.exists(CONVNEXT_PT):
    try:
        with open(STATE_JSON, 'r') as f:
            is_convnext_done = bool(json.load(f).get('completed', False))
    except Exception:
        is_convnext_done = False

if is_convnext_done:
    print(f'ConvNeXt-Small eğitimi önceki oturumda tamamlanmış ({CONVNEXT_PT}), doğrudan yükleniyor!')
    convnext_model.load_state_dict(torch.load(CONVNEXT_PT, map_location=DEVICE))
else:
    print('ConvNeXt-Small (GeM + Quality-MLP) eğitimi başlatılıyor / devam ettiriliyor...')
    ema_conv = ModelEmaV2(convnext_model, decay=0.9995, device=DEVICE)
    if NUM_GPUS > 1:
        convnext_model = nn.DataParallel(convnext_model)

    EPOCHS_CNX = 12
    optimizer = torch.optim.AdamW(convnext_model.parameters(), lr=1.5e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_CNX, eta_min=1e-6)
    scaler = GradScaler()
    weights_t = torch.tensor([1.05, 1.00, 1.15], device=DEVICE)
    y_val_bin = label_binarize(val_labels, classes=[0, 1, 2])
    start_epoch = 1
    best_auc = 0.0

    if os.path.exists(LAST_CKPT):
        print(f'Yarıda kalan ConvNeXt checkpoint bulundu ({LAST_CKPT}), kaldığı yerden devam ediliyor...')
        ckpt = torch.load(LAST_CKPT, map_location=DEVICE)
        raw_m = convnext_model.module if isinstance(convnext_model, nn.DataParallel) else convnext_model
        raw_m.load_state_dict(ckpt['model_state_dict'])
        ema_conv.module.load_state_dict(ckpt['ema_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best_auc = ckpt.get('best_auc', 0.0)
        print(f'Epoch {start_epoch}/{EPOCHS_CNX} noktasından devam ediliyor (Best Val AUC: {best_auc:.4f})')

    for epoch in range(start_epoch, EPOCHS_CNX + 1):
        if time.time() - SESSION_START_TIME > MAX_SESSION_SECONDS:
            print(f'11.2 saat güvenli oturum sınırına ulaşıldı! Epoch {epoch-1} sonunda durduruluyor.')
            break

        convnext_model.train()
        for imgs, q_feats, labels in tqdm(train_loader, desc=f'ConvNeXt Ep {epoch:02d}/{EPOCHS_CNX}', leave=False):
            imgs, q_feats, labels = imgs.to(DEVICE), q_feats.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with autocast():
                out = convnext_model(imgs, q_feats)
                ce = F.cross_entropy(out, labels, reduction='none', label_smoothing=0.05)
                loss = (((1.0 - torch.exp(-ce)) ** 1.5) * ce * weights_t[labels]).mean()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(convnext_model.parameters(), 2.0)
            scaler.step(optimizer); scaler.update()
            ema_conv.update(convnext_model.module if isinstance(convnext_model, nn.DataParallel) else convnext_model)
        scheduler.step()

        eval_m = ema_conv.module
        eval_m.eval()
        vp = []
        with torch.no_grad():
            for imgs, q_feats, _ in val_loader:
                with autocast():
                    vp.append(F.softmax(eval_m(imgs.to(DEVICE), q_feats.to(DEVICE)), dim=1).cpu().numpy())
        vp = np.vstack(vp)
        v_auc = roc_auc_score(y_val_bin, vp, multi_class='ovr', average='macro')
        print(f'ConvNeXt Epoch {epoch:02d} | Val Macro-AUC: {v_auc:.4f}')
        if v_auc > best_auc:
            best_auc = v_auc
            torch.save(eval_m.state_dict(), CONVNEXT_PT)

        raw_m = convnext_model.module if isinstance(convnext_model, nn.DataParallel) else convnext_model
        torch.save({
            'epoch': epoch,
            'model_state_dict': raw_m.state_dict(),
            'ema_state_dict': eval_m.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_auc': best_auc,
        }, LAST_CKPT)

        if epoch == EPOCHS_CNX:
            with open(STATE_JSON, 'w') as f:
                json.dump({'completed': True, 'epoch': epoch, 'best_auc': float(best_auc)}, f)

    if isinstance(convnext_model, nn.DataParallel):
        convnext_model = convnext_model.module
    convnext_model.load_state_dict(torch.load(CONVNEXT_PT, map_location=DEVICE))

convnext_model.eval()


In [ ]:
# ==============================================================================
# Bölüm 4: Çoklu Mimari Hibrit Topluluk Öğrenmesi (Ensemble) ve Bağlamsal Füzyon
# ==============================================================================
def predict_convnext_tta(loader):
    p_list = []
    with torch.no_grad():
        for imgs, q_feats, _ in tqdm(loader, desc='ConvNeXt TTA', leave=False):
            imgs, q_feats = imgs.to(DEVICE), q_feats.to(DEVICE)
            imgs_flip = torch.flip(imgs, dims=[3])
            with autocast():
                p1 = F.softmax(convnext_model(imgs, q_feats), dim=1)
                p2 = F.softmax(convnext_model(imgs_flip, q_feats), dim=1)
            p_list.append((0.55 * p1 + 0.45 * p2).cpu().numpy())
    return np.vstack(p_list)

val_p_cnx  = predict_convnext_tta(val_loader)
test_p_cnx = predict_convnext_tta(test_loader)

# YOLO11m-cls TTA önbelleği ve model ağırlıklarının yüklenmesi
yolo11_cache = glob.glob('/kaggle/input/**/yolo11_tta_cache.npz', recursive=True) + glob.glob('/kaggle/working/**/yolo11_tta_cache.npz', recursive=True)
if yolo11_cache:
    print(f'Hazır YOLO11m-cls TTA önbelleği bulundu: {yolo11_cache[0]}')
    yc = np.load(yolo11_cache[0])
    val_p_y11  = yc['val_s1_tta']
    test_p_y11 = yc['test_s1_tta']
else:
    yolo11_pts = (
        glob.glob('/kaggle/working/**/yolo11*/weights/best.pt', recursive=True) +
        glob.glob('/kaggle/input/**/yolo11*/weights/best.pt', recursive=True)
    )
    if yolo11_pts:
        yolo11_net = YOLO(yolo11_pts[0])
    else:
        print('YOLO11m-cls ağırlığı bulunamadı, hızlı tamamlayıcı eğitim başlatılıyor (12 epoch)...')
        yolo11_net = YOLO('yolo11m-cls.pt')
        yolo11_net.train(data=DATASET_DIR, epochs=12, imgsz=512, batch=32*max(1, NUM_GPUS),
                         project=SAVE_DIR, name='yolo11_ens', exist_ok=True, verbose=False)
        yolo11_net = YOLO(os.path.join(SAVE_DIR, 'yolo11_ens/weights/best.pt'))

    def predict_yolo_tta(file_list, batch_size=64):
        n = len(file_list)
        p_out = np.zeros((n, 3), dtype=np.float64)
        for s in range(0, n, batch_size):
            bf = file_list[s:s+batch_size]
            im_b = [cv2.imread(fp) for fp in bf]
            im_f = [cv2.flip(im, 1) for im in im_b]
            for i, r in enumerate(yolo11_net.predict(im_b, imgsz=512, verbose=False)):
                p_out[s+i] += 0.35 * r.probs.data.cpu().numpy()
            for i, r in enumerate(yolo11_net.predict(im_f, imgsz=512, verbose=False)):
                p_out[s+i] += 0.25 * r.probs.data.cpu().numpy()
            for i, r in enumerate(yolo11_net.predict(im_b, imgsz=576, verbose=False)):
                p_out[s+i] += 0.25 * r.probs.data.cpu().numpy()
            for i, r in enumerate(yolo11_net.predict(im_f, imgsz=576, verbose=False)):
                p_out[s+i] += 0.15 * r.probs.data.cpu().numpy()
        return p_out

    val_p_y11  = predict_yolo_tta(val_samples)
    test_p_y11 = predict_yolo_tta(test_samples)

# Doğrulama kümesi üzerinde optimal ConvNeXt-Small, YOLO11m-cls (ve mevcutsa EfficientNet-B0) topluluk katsayısının optimizasyonu
y_val  = val_labels
y_true = test_labels
y_val_bin = label_binarize(y_val, classes=[0, 1, 2])

effnet_cache = glob.glob('/kaggle/input/**/efficientnet_tta_cache.npz', recursive=True) + glob.glob('/kaggle/working/**/efficientnet_tta_cache.npz', recursive=True)
if effnet_cache:
    ec = np.load(effnet_cache[0])
    val_p_eff, test_p_eff = ec['val_s1_tta'], ec['test_s1_tta']
else:
    val_p_eff, test_p_eff = None, None

best_w_cnx, best_w_y11, best_w_eff, best_ens_auc = 0.55, 0.45, 0.0, -1.0
for w_cnx in np.linspace(0.0, 1.0, 21):
    if val_p_eff is not None:
        for w_eff in np.linspace(0.0, max(0.0, 1.0 - w_cnx), 11):
            w_y11 = max(0.0, 1.0 - w_cnx - w_eff)
            blend_v = w_cnx * val_p_cnx + w_y11 * val_p_y11 + w_eff * val_p_eff
            auc_v = roc_auc_score(y_val_bin, blend_v, multi_class='ovr', average='macro')
            if auc_v > best_ens_auc:
                best_ens_auc, best_w_cnx, best_w_y11, best_w_eff = auc_v, w_cnx, w_y11, w_eff
    else:
        w_y11 = 1.0 - w_cnx
        blend_v = w_cnx * val_p_cnx + w_y11 * val_p_y11
        auc_v = roc_auc_score(y_val_bin, blend_v, multi_class='ovr', average='macro')
        if auc_v > best_ens_auc:
            best_ens_auc, best_w_cnx, best_w_y11, best_w_eff = auc_v, w_cnx, w_y11, 0.0

print(f'Optimal Hibrit Topluluk Ağırlıkları: {best_w_cnx:.2f}*ConvNeXt + {best_w_y11:.2f}*YOLO11 + {best_w_eff:.2f}*EffNet (Val Macro-AUC: {best_ens_auc:.4f})')
val_ens_probs  = best_w_cnx * val_p_cnx  + best_w_y11 * val_p_y11  + (best_w_eff * val_p_eff if val_p_eff is not None else 0.0)
test_ens_probs = best_w_cnx * test_p_cnx + best_w_y11 * test_p_y11 + (best_w_eff * test_p_eff if test_p_eff is not None else 0.0)

# Hasta Düzeyinde Çoklu Projeksiyon Bağlamsal Füzyonu (Patient-Level Multi-View Contextual Fusion)
def apply_patient_multiview_context(probs, cases, views, gamma_mv=0.45):
    """Hasta düzeyinde çoklu projeksiyon (RCC, LCC, RMLO, LMLO) bağlamsal füzyon fonksiyonu.

    Klinik mamografi değerlendirme protokollerine uygun olarak tekil görüntü olasılıklarını,
    ipsilateral, kontralateral ve bilateral projeksiyon havuzlama çıktıları ile harmanlar.
    """
    if (cases >= 0).sum() < len(cases) * 0.5:
        return probs
    out_p = probs.copy()
    df_c = pd.DataFrame({'case': cases, 'view': views, 'p0': probs[:, 0], 'p1': probs[:, 1], 'p2': probs[:, 2]})
    case_mean = df_c.groupby('case')[['p0', 'p1', 'p2']].transform('mean').values
    case_max  = df_c.groupby('case')[['p0', 'p1', 'p2']].transform('max').values

    patient_context = np.zeros_like(probs)
    patient_context[:, 0] = 0.70 * case_mean[:, 0] + 0.30 * case_max[:, 0]
    patient_context[:, 1] = 0.75 * case_mean[:, 1] + 0.25 * case_max[:, 1]
    patient_context[:, 2] = 0.45 * case_mean[:, 2] + 0.55 * case_max[:, 2]
    patient_context /= np.clip(patient_context.sum(axis=1, keepdims=True), 1e-8, None)

    fused = (1.0 - gamma_mv) * out_p + gamma_mv * patient_context
    return fused / np.clip(fused.sum(axis=1, keepdims=True), 1e-8, None)

best_gamma, best_mv_auc = 0.0, best_ens_auc
for g_cand in np.linspace(0.0, 0.75, 16):
    cand_v = apply_patient_multiview_context(val_ens_probs, val_cases, val_views, gamma_mv=g_cand)
    c_auc = roc_auc_score(y_val_bin, cand_v, multi_class='ovr', average='macro')
    if c_auc > best_mv_auc:
        best_mv_auc = c_auc
        best_gamma = g_cand

print(f'Optimal Multi-View Bağlam Ağırlığı: gamma={best_gamma:.2f} (Val Macro AUC: {best_ens_auc:.4f} -> {best_mv_auc:.4f})')
val_probs_mv  = apply_patient_multiview_context(val_ens_probs, val_cases, val_views, gamma_mv=best_gamma)
test_probs_mv = apply_patient_multiview_context(test_ens_probs, test_cases, test_views, gamma_mv=best_gamma)

def apply_calibration(probs, params):
    w0, w1, w2 = params
    logits = np.log(np.clip(probs, 1e-8, 1.0))
    logits[:, 0] += w0; logits[:, 1] += w1; logits[:, 2] += w2
    exp_l = np.exp(logits - logits.max(axis=1, keepdims=True))
    return exp_l / exp_l.sum(axis=1, keepdims=True)

def optimize_pareto_calibration(val_p, y_v):
    """Klinik karar eşiklerinin Pareto-optimal kalibrasyonu.

    Yüksek riskli BI-RADS 4-5 sınıfında yanlış negatifliği asgariye indirmek amacıyla
    duyarlılık (sensitivity) ile genel makro F1 skorunu eşzamanlı optimize eder.
    """
    raw_preds_v = val_p.argmax(axis=1)
    raw_rec_45  = ((raw_preds_v == 2) & (y_v == 2)).sum() / max((y_v == 2).sum(), 1)
    raw_f1_0    = f1_score(y_v, raw_preds_v, average=None)[0]
    def _obj(params):
        cal_v = apply_calibration(val_p, params)
        preds_v = cal_v.argmax(axis=1)
        f1_c = f1_score(y_v, preds_v, average=None)
        tp_45 = ((preds_v == 2) & (y_v == 2)).sum(); fn_45 = ((preds_v != 2) & (y_v == 2)).sum()
        tn_45 = ((preds_v != 2) & (y_v != 2)).sum(); fp_45 = ((preds_v == 2) & (y_v != 2)).sum()
        rec_45  = tp_45 / max(tp_45 + fn_45, 1)
        spec_45 = tn_45 / max(tn_45 + fp_45, 1)
        penalty = 0.0
        if rec_45 < raw_rec_45 + 0.015: penalty += 6.0 * (raw_rec_45 + 0.015 - rec_45)
        if f1_c[0] < raw_f1_0: penalty += 5.0 * (raw_f1_0 - f1_c[0])
        return -(0.40 * f1_c.mean() + 0.30 * accuracy_score(y_v, preds_v) + 0.30 * 0.5 * (rec_45 + spec_45)) + penalty + 0.015 * sum(p**2 for p in params)
    best_p, best_val = [0.0, 0.0, 0.0], _obj([0.0, 0.0, 0.0])
    for iw0 in [-0.06, 0.0, 0.06]:
        for iw2 in [0.0, 0.08, 0.16]:
            res = minimize(_obj, x0=[iw0, 0.0, iw2], method='Nelder-Mead', options={'maxiter': 500})
            if res.fun < best_val: best_val, best_p = res.fun, res.x
    return best_p

best_params = optimize_pareto_calibration(val_probs_mv, y_val)
y_scores = apply_calibration(test_probs_mv, best_params)
y_pred   = y_scores.argmax(axis=1)
acc      = (y_true == y_pred).mean() * 100

print(f'\n SOTA Hibrit Ensemble + Multi-View Test Doğruluğu: {acc:.2f}%')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))


In [ ]:
# ==============================================================================
# Bölüm 5: Bütüncül Model Karşılaştırması ve Ablasyon Analizi (Master Tablo 4)
# ==============================================================================
def compute_paper_table4_metrics(y_t, probs, preds=None):
    """Makale Tablo 4 için tanısal ve istatistiksel performans metriklerini hesaplar.

    Metrikler: Doğruluk, Makro/Sınıf Bazlı Hassasiyet, Duyarlılık, F1-Skoru, Makro ROC-AUC
    ve klinik sevk odaklı BI-RADS 4-5 duyarlılık oranı.
    """
    if preds is None: preds = probs.argmax(axis=1)
    y_bin = label_binarize(y_t, classes=[0, 1, 2])
    diag_mask = (y_t != 0) & (preds != 0)
    diag_acc = (y_t[diag_mask] == preds[diag_mask]).mean() * 100.0 if diag_mask.sum() > 0 else 0.0
    tp_45 = ((preds == 2) & (y_t == 2)).sum(); fn_45 = ((preds != 2) & (y_t == 2)).sum()
    tn_45 = ((preds != 2) & (y_t != 2)).sum(); fp_45 = ((preds == 2) & (y_t != 2)).sum()
    rec_45  = (tp_45 / max(tp_45 + fn_45, 1)) * 100.0
    spec_45 = (tn_45 / max(tn_45 + fp_45, 1)) * 100.0
    prec_45 = (tp_45 / max(tp_45 + fp_45, 1)) * 100.0
    f1_c = f1_score(y_t, preds, average=None)
    return {
        'BI-RADS 4-5 ROC-AUC': round(roc_auc_score(y_bin[:, 2], probs[:, 2]), 4),
        'BI-RADS 1-2 ROC-AUC': round(roc_auc_score(y_bin[:, 1], probs[:, 1]), 4),
        'BI-RADS 0 ROC-AUC': round(roc_auc_score(y_bin[:, 0], probs[:, 0]), 4),
        'ROC-AUC (Macro)': round(roc_auc_score(y_bin, probs, multi_class='ovr', average='macro'), 4),
        '2x2 Tanısal Alt-Matris Doğruluğu (1-2 vs 4-5) (%)': round(diag_acc, 2),
        'BI-RADS 4-5 Dengeli Doğruluk (%)': round(0.5 * (rec_45 + spec_45), 2),
        'BI-RADS 4-5 Özgüllük / Specificity (%)': round(spec_45, 2),
        'BI-RADS 4-5 Kesinlik / Precision (%)': round(prec_45, 2),
        'BI-RADS 4-5 Duyarlılık / Recall (%)': round(rec_45, 2),
        'BI-RADS 4-5 F1-Skoru': round(f1_c[2], 4),
        'BI-RADS 1-2 F1-Skoru': round(f1_c[1], 4),
        'BI-RADS 0 F1-Skoru': round(f1_c[0], 4),
        '3-Sınıflı Genel Doğruluk (Test Accuracy) (%)': round((y_t == preds).mean() * 100.0, 2),
        'Makro F1-Skoru (3-Sınıf)': round(f1_score(y_t, preds, average='macro'), 4),
    }

yolo11_paper_base = {
    'BI-RADS 4-5 ROC-AUC': 0.8090, 'BI-RADS 1-2 ROC-AUC': 0.7790, 'BI-RADS 0 ROC-AUC': 0.7180,
    'ROC-AUC (Macro)': 0.7690, '2x2 Tanısal Alt-Matris Doğruluğu (1-2 vs 4-5) (%)': 77.19,
    'BI-RADS 4-5 Dengeli Doğruluk (%)': 71.80, 'BI-RADS 4-5 Özgüllük / Specificity (%)': 86.22,
    'BI-RADS 4-5 Kesinlik / Precision (%)': 72.19, 'BI-RADS 4-5 Duyarlılık / Recall (%)': 57.37,
    'BI-RADS 4-5 F1-Skoru': 0.6400, 'BI-RADS 1-2 F1-Skoru': 0.6500, 'BI-RADS 0 F1-Skoru': 0.4200,
    '3-Sınıflı Genel Doğruluk (Test Accuracy) (%)': 59.40, 'Makro F1-Skoru (3-Sınıf)': 0.5700,
}

master_df = pd.DataFrame({
    'Makale Baz (YOLO11m-cls)': yolo11_paper_base,
    'YOLO11m-cls (6-Yön TTA)': compute_paper_table4_metrics(y_true, test_p_y11),
    'ConvNeXt-Small (GeM+Q-MLP)': compute_paper_table4_metrics(y_true, test_p_cnx),
    'SOTA Ensemble + Multi-View': compute_paper_table4_metrics(y_true, y_scores, y_pred),
})
master_df['Net Kazanç (Ensemble - Baz)'] = (master_df['SOTA Ensemble + Multi-View'] - master_df['Makale Baz (YOLO11m-cls)']).round(4)

print('\n' + '=' * 115)
print('TABLO 4 — MODEL PERFORMANS KARŞILAŞTIRMASI')
print('=' * 115)
print(master_df.to_string())
print('=' * 115)
master_df.to_csv(os.path.join(SAVE_DIR, 'sota_ensemble_master_table4.csv'))


In [ ]:
# ==============================================================================
# Bölüm 6: Açıklanabilir Yapay Zeka (XAI) ve Grad-CAM Aktivasyon Haritaları
# ==============================================================================
def generate_gradcam(model_cnx, img_tensor, q_tensor, target_class):
    """Grad-CAM (Gradient-weighted Class Activation Mapping) Açıklanabilirlik Analizi.

    Hedef sınıf için son evrişimsel katman gradyanları ile aktivasyon haritalarının ağırlıklı
    kombinasyonunu oluşturarak karar bölgelerini ısı haritası olarak çıkarır.
    """
    model_cnx.eval()
    activations, gradients = [], []

    def fwd_hook(module, inp, out): activations.append(out.detach())
    def bwd_hook(module, grad_in, grad_out): gradients.append(grad_out[0].detach())

    target_layer = model_cnx.backbone.stages[-1]
    h1 = target_layer.register_forward_hook(fwd_hook)
    h2 = target_layer.register_full_backward_hook(bwd_hook)

    img_in = img_tensor.unsqueeze(0).to(DEVICE).requires_grad_(True)
    q_in   = q_tensor.unsqueeze(0).to(DEVICE)
    logits = model_cnx(img_in, q_in)
    model_cnx.zero_grad()
    logits[0, target_class].backward()

    h1.remove(); h2.remove()
    act  = activations[0][0]
    grad = gradients[0][0]
    weights = grad.mean(dim=(1, 2), keepdim=True)
    cam = F.relu((weights * act).sum(dim=0)).cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-7)
    return cv2.resize(cam, (IMG_SIZE, IMG_SIZE))

fig, axes = plt.subplots(3, 3, figsize=(13, 12))
fig.patch.set_facecolor('#1a1a2e')

for cls_idx, cls_title in enumerate(DISPLAY_NAMES):
    correct_indices = np.where((y_true == cls_idx) & (y_pred == cls_idx))[0]
    if len(correct_indices) == 0:
        correct_indices = np.where(y_true == cls_idx)[0]
    best_idx = correct_indices[np.argmax(y_scores[correct_indices, cls_idx])]

    img_t, q_t, _ = test_ds[best_idx]
    cam = generate_gradcam(convnext_model, img_t, q_t, cls_idx)
    raw_bgr = cv2.resize(cv2.imread(test_samples[best_idx]), (IMG_SIZE, IMG_SIZE))
    raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
    heatmap = cv2.cvtColor(cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(raw_rgb, 0.58, heatmap, 0.42, 0)

    axes[cls_idx, 0].imshow(raw_rgb)
    axes[cls_idx, 0].set_title(f'{cls_title}\nOrijinal Önişlenmiş Mamogram', color='white', fontsize=11)
    axes[cls_idx, 1].imshow(cam, cmap='jet')
    axes[cls_idx, 1].set_title(f'Grad-CAM Aktivasyon Haritası', color='white', fontsize=11)
    axes[cls_idx, 2].imshow(overlay)
    axes[cls_idx, 2].set_title(f'Lezyon/Bölge Odak Örtüşmesi (p={y_scores[best_idx, cls_idx]:.2f})', color='white', fontsize=11)
    for j in range(3): axes[cls_idx, j].axis('off')

plt.suptitle('MammosighTR — Grad-CAM Görsel Açıklanabilirlik Analizi (BI-RADS 0 / 1-2 / 4-5)',
             color='white', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
gradcam_path = '/kaggle/working/mammosight_gradcam.png' if os.path.exists('/kaggle') else './mammosight_gradcam.png'
plt.savefig(gradcam_path, dpi=180, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Grad-CAM görseli kaydedildi: {gradcam_path}')
